In [1]:
import json, numpy as np, pandas as pd, os, warnings
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from scipy.stats import rankdata
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

REF_DATE = pd.Timestamp('2021-01-01')

# ============================================================
# DATA LOADING
# ============================================================
def load_nested_json(path):
    with open(path) as f:
        raw = json.load(f)
    records = [rec for applicant in raw for rec in applicant if rec is not None]
    return pd.DataFrame(records)

# ============================================================
# 1. PAYMENT HISTORY PARSING (V2)
# ============================================================
def _parse_ph(s):
    s = str(s)
    if s in ('', 'nan') or len(s) < 3:
        return []
    n = len(s) - (len(s) % 3)
    return [int(s[i:i+3]) for i in range(0, n, 3)]

def _ph_row_features_v2(s):
    vals = _parse_ph(s)
    base = dict(
        ph_n_months=0, ph_max_dpd=0, ph_mean_dpd=0, ph_sum_dpd=0,
        ph_n_late=0, ph_n_30=0, ph_n_90=0, ph_n_180=0, ph_n_360=0,
        ph_recent_dpd=0, ph_recent3_max=0, ph_recent6_max=0,
        ph_recent12_max=0, ph_late_ratio=0, ph_has_history=0,
        ph_first_half_late=0, ph_second_half_late=0,
        ph_trend=0, ph_max_consecutive_late=0,
        ph_months_since_last_late=-1,  # -1 = never late (not 999)
        ph_late_in_last_6=0, ph_late_in_last_12=0,
    )
    if not vals:
        return base

    arr = np.array(vals)
    n = len(arr)
    late = (arr > 0)

    base['ph_n_months'] = n
    base['ph_max_dpd'] = int(arr.max())
    base['ph_mean_dpd'] = float(arr.mean())
    base['ph_sum_dpd'] = int(arr.sum())
    base['ph_n_late'] = int(late.sum())
    base['ph_n_30'] = int((arr >= 30).sum())
    base['ph_n_90'] = int((arr >= 90).sum())
    base['ph_n_180'] = int((arr >= 180).sum())
    base['ph_n_360'] = int((arr >= 360).sum())
    base['ph_late_ratio'] = float(late.mean())
    base['ph_has_history'] = 1

    base['ph_recent_dpd'] = int(arr[-1])
    base['ph_recent3_max'] = int(arr[-3:].max())
    base['ph_recent6_max'] = int(arr[-6:].max()) if n >= 6 else int(arr.max())
    base['ph_recent12_max'] = int(arr[-12:].max()) if n >= 12 else int(arr.max())

    base['ph_late_in_last_6'] = int(late[-6:].sum()) if n >= 6 else int(late.sum())
    base['ph_late_in_last_12'] = int(late[-12:].sum()) if n >= 12 else int(late.sum())

    if n >= 4:
        mid = n // 2
        first_rate = late[:mid].mean()
        second_rate = late[mid:].mean()
        base['ph_first_half_late'] = float(first_rate)
        base['ph_second_half_late'] = float(second_rate)
        base['ph_trend'] = float(second_rate - first_rate)

    if late.any():
        max_consec = current = 0
        for l in late:
            if l:
                current += 1
                max_consec = max(max_consec, current)
            else:
                current = 0
        base['ph_max_consecutive_late'] = max_consec
        late_indices = np.where(late)[0]
        base['ph_months_since_last_late'] = n - 1 - late_indices[-1]

    return base

def build_payment_features_v2(acc):
    ph = acc['payment_hist_string'].apply(_ph_row_features_v2).apply(pd.Series)
    ph['uid'] = acc['uid'].values

    g = ph.groupby('uid')
    out = g.agg(
        pmt_max_dpd          = ('ph_max_dpd', 'max'),
        pmt_mean_dpd         = ('ph_mean_dpd', 'mean'),
        pmt_sum_dpd          = ('ph_sum_dpd', 'sum'),
        pmt_n_late           = ('ph_n_late', 'sum'),
        pmt_n_30             = ('ph_n_30', 'sum'),
        pmt_n_90             = ('ph_n_90', 'sum'),
        pmt_n_180            = ('ph_n_180', 'sum'),
        pmt_n_360            = ('ph_n_360', 'sum'),
        pmt_recent_dpd_max   = ('ph_recent_dpd', 'max'),
        pmt_recent3_max      = ('ph_recent3_max', 'max'),
        pmt_recent6_max      = ('ph_recent6_max', 'max'),
        pmt_recent12_max     = ('ph_recent12_max', 'max'),
        pmt_late_last6_sum   = ('ph_late_in_last_6', 'sum'),
        pmt_late_last12_sum  = ('ph_late_in_last_12', 'sum'),
        pmt_trend_max        = ('ph_trend', 'max'),
        pmt_trend_mean       = ('ph_trend', 'mean'),
        pmt_max_consec_late  = ('ph_max_consecutive_late', 'max'),
        pmt_total_months     = ('ph_n_months', 'sum'),
        pmt_late_ratio_max   = ('ph_late_ratio', 'max'),
        pmt_late_ratio_mean  = ('ph_late_ratio', 'mean'),
        pmt_n_acc_with_hist  = ('ph_has_history', 'sum'),
    )

    # FIX #2: months_since_late must be the MOST RECENT late event across
    # a user's accounts = minimum non-negative value. Using 'max' previously
    # picked the OLDEST delinquency (or the -1 sentinel), which is misleading.
    # -1 = never late; 0 = late last month (worst); larger = longer ago.
    def _min_nonneg_or_default(series, default=-1):
        nonneg = series[series >= 0]
        if len(nonneg) == 0:
            return default
        return int(nonneg.min())

    out['pmt_months_since_late'] = (
        ph.groupby('uid')['ph_months_since_last_late']
          .agg(_min_nonneg_or_default)
    )

    out['pmt_ever_90plus']  = (out['pmt_n_90'] > 0).astype(int)
    out['pmt_ever_180plus'] = (out['pmt_n_180'] > 0).astype(int)
    out['pmt_ever_360plus'] = (out['pmt_n_360'] > 0).astype(int)
    out['pmt_recent_late_ratio'] = out['pmt_late_last6_sum'] / (out['pmt_n_late'] + 1)

    return out.reset_index()

# ============================================================
# 2. ACCOUNT FEATURES (V2)
# ============================================================
KEEP_CREDIT_TYPES = ['Consumer credit', 'Credit card', 'Car loan',
                     'Mortgage', 'Microloan']

def build_account_features_v2(acc):
    acc = acc.copy()
    acc['open_date']      = pd.to_datetime(acc['open_date'], errors='coerce')
    acc['closed_date']    = pd.to_datetime(acc['closed_date'], errors='coerce')
    acc['loan_amount']    = pd.to_numeric(acc['loan_amount'], errors='coerce')
    acc['amount_overdue'] = pd.to_numeric(acc['amount_overdue'], errors='coerce')

    bad = acc['closed_date'] < acc['open_date']
    acc.loc[bad, 'closed_date'] = pd.NaT

    acc['is_open']          = acc['closed_date'].isna().astype(int)
    acc['is_zero_amt']      = (acc['loan_amount'] == 0).astype(int)
    acc['has_overdue']      = (acc['amount_overdue'] > 0).astype(int)
    acc['days_since_open']  = (REF_DATE - acc['open_date']).dt.days
    acc['days_since_close'] = (REF_DATE - acc['closed_date']).dt.days
    acc['log_loan']         = np.log1p(acc['loan_amount'].clip(lower=0))
    acc['account_age']      = (REF_DATE - acc['open_date']).dt.days
    acc['loan_duration']    = (acc['closed_date'] - acc['open_date']).dt.days
    acc['is_recent']        = (acc['days_since_open'] <= 365).astype(int)
    acc['is_very_recent']   = (acc['days_since_open'] <= 180).astype(int)

    acc['credit_grp'] = np.where(
        acc['credit_type'].isin(KEEP_CREDIT_TYPES),
        acc['credit_type'], 'Other')

    g = acc.groupby('uid')

    num = g.agg(
        acc_n_accounts          = ('uid', 'size'),
        acc_n_open              = ('is_open', 'sum'),
        acc_n_zero_amt          = ('is_zero_amt', 'sum'),
        acc_loan_amt_sum        = ('loan_amount', 'sum'),
        acc_loan_amt_mean       = ('loan_amount', 'mean'),
        acc_loan_amt_max        = ('loan_amount', 'max'),
        acc_loan_amt_std        = ('loan_amount', 'std'),
        acc_loan_amt_median     = ('loan_amount', 'median'),
        acc_log_loan_sum        = ('log_loan', 'sum'),
        acc_log_loan_mean       = ('log_loan', 'mean'),
        acc_overdue_sum         = ('amount_overdue', 'sum'),
        acc_overdue_max         = ('amount_overdue', 'max'),
        acc_n_with_overdue      = ('has_overdue', 'sum'),
        acc_n_credit_types      = ('credit_type', 'nunique'),
        acc_days_since_open_min = ('days_since_open', 'min'),
        acc_days_since_open_max = ('days_since_open', 'max'),
        acc_days_since_open_mean= ('days_since_open', 'mean'),
        acc_days_since_close_min= ('days_since_close', 'min'),
        acc_n_recent            = ('is_recent', 'sum'),
        acc_n_very_recent       = ('is_very_recent', 'sum'),
        acc_age_min             = ('account_age', 'min'),
        acc_age_max             = ('account_age', 'max'),
        acc_duration_mean       = ('loan_duration', 'mean'),
        acc_duration_min        = ('loan_duration', 'min'),
    )

    num['acc_open_ratio']        = num['acc_n_open'] / num['acc_n_accounts']
    num['acc_overdue_ratio']     = num['acc_n_with_overdue'] / num['acc_n_accounts']
    num['acc_overdue_amt_ratio'] = num['acc_overdue_sum'] / (num['acc_loan_amt_sum'] + 1)
    num['acc_zero_amt_ratio']    = num['acc_n_zero_amt'] / num['acc_n_accounts']
    num['acc_recent_ratio']      = num['acc_n_recent'] / num['acc_n_accounts']
    num['acc_age_range']         = num['acc_age_max'] - num['acc_age_min']

    cat = (acc.groupby(['uid', 'credit_grp']).size()
              .unstack(fill_value=0).add_prefix('acc_cnt_'))
    cat_share = cat.div(cat.sum(axis=1), axis=0).add_suffix('_share')

    open_acc = acc[acc['is_open'] == 1]
    if len(open_acc) > 0:
        open_agg = open_acc.groupby('uid').agg(
            acc_open_loan_sum  = ('loan_amount', 'sum'),
            acc_open_loan_mean = ('loan_amount', 'mean'),
            acc_open_loan_max  = ('loan_amount', 'max'),
        )
    else:
        open_agg = pd.DataFrame(
            columns=['acc_open_loan_sum','acc_open_loan_mean','acc_open_loan_max'])

    out = num.join(cat).join(cat_share).join(open_agg)
    return out.reset_index()

# ============================================================
# 3. ENQUIRY FEATURES (V2)
# ============================================================
def build_enquiry_features_v2(enq):
    enq = enq.copy()
    enq['enquiry_date'] = pd.to_datetime(enq['enquiry_date'], errors='coerce')
    enq['enquiry_amt']  = pd.to_numeric(enq['enquiry_amt'], errors='coerce')
    enq['log_enq_amt']  = np.log1p(enq['enquiry_amt'].clip(lower=0))
    enq['days_ago']     = (REF_DATE - enq['enquiry_date']).dt.days

    for w in [30, 90, 180, 365]:
        enq[f'in_{w}d'] = (enq['days_ago'] <= w).astype(int)

    enq['is_real_type'] = enq['enquiry_type'].isin(
        ['Cash loans', 'Revolving loans']).astype(int)

    g = enq.groupby('uid')

    num = g.agg(
        enq_n                = ('uid', 'size'),
        enq_amt_sum          = ('enquiry_amt', 'sum'),
        enq_amt_mean         = ('enquiry_amt', 'mean'),
        enq_amt_max          = ('enquiry_amt', 'max'),
        enq_amt_std          = ('enquiry_amt', 'std'),
        enq_amt_median       = ('enquiry_amt', 'median'),
        enq_log_amt_sum      = ('log_enq_amt', 'sum'),
        enq_log_amt_mean     = ('log_enq_amt', 'mean'),
        enq_n_types          = ('enquiry_type', 'nunique'),
        enq_days_since_last  = ('days_ago', 'min'),
        enq_days_since_first = ('days_ago', 'max'),
        enq_n_30d            = ('in_30d', 'sum'),
        enq_n_90d            = ('in_90d', 'sum'),
        enq_n_180d           = ('in_180d', 'sum'),
        enq_n_365d           = ('in_365d', 'sum'),
        enq_n_real           = ('is_real_type', 'sum'),
    )

    num['enq_span_days']     = num['enq_days_since_first'] - num['enq_days_since_last']
    num['enq_real_ratio']    = num['enq_n_real'] / (num['enq_n'] + 1e-6)
    num['enq_velocity_30d']  = num['enq_n_30d'] / 30
    num['enq_velocity_90d']  = num['enq_n_90d'] / 90
    num['enq_velocity_365d'] = num['enq_n_365d'] / 365
    num['enq_accel']         = num['enq_n_90d'] / (num['enq_n_365d'] + 1) * 4

    # Recent amount mean
    recent_enq = enq[enq['days_ago'] <= 180]
    if len(recent_enq) > 0:
        recent_amt = recent_enq.groupby('uid')['enquiry_amt'].mean()
        recent_amt.name = 'enq_recent_amt_mean'
        num = num.join(recent_amt)
    else:
        num['enq_recent_amt_mean'] = np.nan

    # Categorical counts
    KEEP_ENQ = ['Cash loans', 'Revolving loans']
    enq['enq_grp'] = np.where(enq['enquiry_type'].isin(KEEP_ENQ),
                              enq['enquiry_type'], 'Other')
    cat = (enq.groupby(['uid', 'enq_grp']).size()
              .unstack(fill_value=0).add_prefix('enq_cnt_'))

    # Inter-enquiry gap stats
    enq_sorted = enq.sort_values(['uid', 'enquiry_date'])
    enq_sorted['prev_date'] = enq_sorted.groupby('uid')['enquiry_date'].shift(1)
    enq_sorted['gap_days'] = (
        enq_sorted['enquiry_date'] - enq_sorted['prev_date']
    ).dt.days

    gap_stats = enq_sorted.groupby('uid')['gap_days'].agg(
        enq_gap_mean = 'mean',
        enq_gap_min  = 'min',
        enq_gap_std  = 'std',
    )

    out = num.join(cat).join(gap_stats)
    return out.reset_index()


# ============================================================
# 4. CROSS-TABLE INTERACTION FEATURES
# ============================================================
def build_cross_features(df):
    df = df.copy()
    eps = 1e-6

    df['enq_to_acc_ratio']     = df['enq_n'] / (df['acc_n_accounts'] + 1)
    df['enq_amt_vs_loan']      = df['enq_amt_mean'] / (df['acc_loan_amt_mean'] + eps)
    df['recent_enq_vs_acc']    = df['enq_n_90d'] / (df['acc_n_recent'] + 1)
    df['open_burden']          = df['acc_open_ratio'] * df['acc_log_loan_mean']
    df['tenure_vs_enq_span']   = df['acc_days_since_open_mean'] / (df['enq_span_days'] + 30)
    df['velocity_per_account'] = df['enq_velocity_90d'] / (df['acc_n_accounts'] + 1)
    df['stress_x_enquiry']     = df['pmt_late_ratio_mean'] * df['enq_n_90d']
    df['overdue_x_enquiry']    = df['acc_n_with_overdue'] * df['enq_n']
    df['recent_dpd_x_enq']     = df['pmt_recent6_max'] * df['enq_n_180d']
    df['newest_acc_age_inv']   = 1.0 / (df['acc_days_since_open_min'] + 1)

    # FIX #1: safe access for a column that may not exist in test.
    # 'acc_cnt_Microloan_share' is only created if Microloan accounts appear.
    # This runs before the Step 7 column-alignment guard, so handle it here.
    if 'acc_cnt_Microloan_share' in df.columns:
        df['microloan_x_open'] = df['acc_cnt_Microloan_share'] * df['acc_open_ratio']
    else:
        df['microloan_x_open'] = 0.0

    return df


# ============================================================
# 5. TARGET ENCODING (FIXED)
# ============================================================
def add_target_encoding(train_df, test_df, y, col, cv, smoothing=20):
    global_mean = y.mean()
    train_df = train_df.copy()
    test_df = test_df.copy()

    new_col = f'{col}_te'
    train_df[new_col] = np.nan

    for tr_idx, val_idx in cv.split(train_df, y):
        tr_y = y.iloc[tr_idx]
        tr_vals = train_df[col].iloc[tr_idx]

        fold_df = pd.DataFrame({'col': tr_vals.values, 'target': tr_y.values})
        stats = fold_df.groupby('col')['target'].agg(['sum', 'count'])
        stats.columns = ['pos', 'n']
        stats['te'] = (stats['pos'] + smoothing * global_mean) / (stats['n'] + smoothing)

        train_df.iloc[val_idx, train_df.columns.get_loc(new_col)] = (
            train_df[col].iloc[val_idx].map(stats['te'])
        )

    train_df[new_col] = train_df[new_col].fillna(global_mean)

    # Test: use full training data
    full_df = pd.DataFrame({'col': train_df[col].values, 'target': y.values})
    stats_full = full_df.groupby('col')['target'].agg(['sum', 'count'])
    stats_full.columns = ['pos', 'n']
    stats_full['te'] = (stats_full['pos'] + smoothing * global_mean) / (stats_full['n'] + smoothing)
    test_df[new_col] = test_df[col].map(stats_full['te']).fillna(global_mean)

    return train_df, test_df


# ============================================================
# 6. MASTER BUILD (V2)
# ============================================================
def build_master_v2(flag, acc, enq):
    m = flag.copy()
    m['is_cash_loan'] = (m['NAME_CONTRACT_TYPE'] == 'Cash loans').astype(int)

    m = m.merge(build_account_features_v2(acc), on='uid', how='left')
    m = m.merge(build_payment_features_v2(acc), on='uid', how='left')
    m = m.merge(build_enquiry_features_v2(enq), on='uid', how='left')

    m['has_accounts'] = m['acc_n_accounts'].notna().astype(int)

    acc_cols = [c for c in m.columns if c.startswith(('acc_', 'pmt_'))]
    m[acc_cols] = m[acc_cols].fillna(0)
    enq_cols = [c for c in m.columns if c.startswith('enq_')]
    m[enq_cols] = m[enq_cols].fillna(0)
    m = m.fillna(0)

    m = build_cross_features(m)
    return m


# ============================================================
# 7. LOAD DATA & BUILD FEATURES
# ============================================================
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

flag_train = pd.read_csv(r'E:\senior_ds_test\senior_ds_test\data\train\train_flag.csv')
acc_train  = load_nested_json(r'E:\senior_ds_test\senior_ds_test\data\train\accounts_data_train.json')
enq_train  = load_nested_json(r'E:\senior_ds_test\senior_ds_test\data\train\enquiry_data_train.json')

flag_test  = pd.read_csv(r'E:\senior_ds_test\senior_ds_test\data\test\test_flag.csv')
acc_test   = load_nested_json(r'E:\senior_ds_test\senior_ds_test\data\test\accounts_data_test.json')
enq_test   = load_nested_json(r'E:\senior_ds_test\senior_ds_test\data\test\enquiry_data_test.json')

print(f"  flag_train: {flag_train.shape}")
print(f"  acc_train:  {acc_train.shape}  ({acc_train['uid'].nunique()} uids)")
print(f"  enq_train:  {enq_train.shape}  ({enq_train['uid'].nunique()} uids)")

print("\nBuilding features...")
train = build_master_v2(flag_train, acc_train, enq_train)
test  = build_master_v2(flag_test, acc_test, enq_test)

# Target encoding
y  = train['TARGET']
cv = StratifiedKFold(5, shuffle=True, random_state=42)
train, test = add_target_encoding(train, test, y, 'NAME_CONTRACT_TYPE', cv)

# Align columns
feature_cols = [c for c in train.columns
                if c not in ['uid', 'TARGET', 'NAME_CONTRACT_TYPE']]
for c in feature_cols:
    if c not in test.columns:
        test[c] = 0
extra = [c for c in test.columns if c not in ['uid'] + feature_cols]
test = test.drop(columns=extra, errors='ignore')
test = test[['uid'] + feature_cols]

print(f"\n  Train: {train.shape}")
print(f"  Test:  {test.shape}")
print(f"  Features: {len(feature_cols)}")
print(f"  NaN train: {train[feature_cols].isna().any().any()}")
print(f"  NaN test:  {test[feature_cols].isna().any().any()}")
print(f"  has_accounts mean: {train['has_accounts'].mean():.4f}")


# ============================================================
# 8. STEP 1: BASELINE
# ============================================================
print("\n" + "=" * 60)
print("STEP 1: Baseline with all features")
print("=" * 60)

X = train[feature_cols]
y = train['TARGET']

BEST_PARAMS = dict(
    n_estimators=600, max_depth=3, learning_rate=0.05,
    subsample=0.74, colsample_bytree=0.65,
    min_child_weight=8, gamma=3.86,
    reg_alpha=3.67, reg_lambda=8.80,
    scale_pos_weight=1.0,
    eval_metric='auc', random_state=42, n_jobs=-1,
)

xgb_base = XGBClassifier(**BEST_PARAMS)
oof_base = cross_val_predict(xgb_base, X, y, cv=cv,
                             method='predict_proba', n_jobs=-1)[:, 1]
auc_base = roc_auc_score(y, oof_base)
print(f"  All {len(feature_cols)} features: AUC = {auc_base:.5f}")

# Feature importance
xgb_base.fit(X, y)
imp = pd.Series(xgb_base.feature_importances_,
                index=feature_cols).sort_values(ascending=False)
print("\n  Top 25 features:")
for i, (feat, val) in enumerate(imp.head(25).items()):
    print(f"    {i+1:2d}. {feat:40s} {val:.4f}")


# ============================================================
# 9. STEP 2: FEATURE SELECTION
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Feature selection")
print("=" * 60)

best_auc  = auc_base
best_cols = feature_cols.copy()

for q in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    threshold = imp.quantile(q)
    keep = imp[imp > threshold].index.tolist()
    oof = cross_val_predict(
        XGBClassifier(**BEST_PARAMS), train[keep], y,
        cv=cv, method='predict_proba', n_jobs=-1
    )[:, 1]
    auc = roc_auc_score(y, oof)
    marker = ' ✓ NEW BEST' if auc > best_auc else ''
    print(f"  Drop bottom {int(q*100):2d}% -> {len(keep):3d} feats: "
          f"AUC={auc:.5f}{marker}")
    if auc > best_auc:
        best_auc, best_cols = auc, keep

print(f"\n  Selected: {len(best_cols)} features, AUC={best_auc:.5f}")


# ============================================================
# 10. STEP 3: OPTUNA TUNING
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Hyperparameter tuning (Optuna, 60 trials)")
print("=" * 60)

Xsel = train[best_cols]

def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 400, 1000),
        max_depth        = trial.suggest_int('max_depth', 2, 5),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        subsample        = trial.suggest_float('subsample', 0.6, 0.95),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 0.9),
        min_child_weight = trial.suggest_int('min_child_weight', 5, 30),
        gamma            = trial.suggest_float('gamma', 1, 8),
        reg_alpha        = trial.suggest_float('reg_alpha', 1, 8),
        reg_lambda       = trial.suggest_float('reg_lambda', 3, 15),
        scale_pos_weight = trial.suggest_float('scale_pos_weight', 0.5, 3.0),
        eval_metric='auc', random_state=42, n_jobs=-1,
    )
    m = XGBClassifier(**params)
    oof = cross_val_predict(m, Xsel, y, cv=cv,
                            method='predict_proba', n_jobs=-1)[:, 1]
    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f"\n  Best Optuna AUC: {study.best_value:.5f}")
print(f"  Best params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

FINAL_PARAMS = {
    **study.best_params,
    'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1
}


# ============================================================
# 11. STEP 4: MULTI-MODEL ENSEMBLE
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Multi-model ensemble")
print("=" * 60)

best_spw = FINAL_PARAMS.get('scale_pos_weight', 1.0)

# --- XGBoost ---
xgb_final = XGBClassifier(**FINAL_PARAMS)
oof_xgb = cross_val_predict(xgb_final, Xsel, y, cv=cv,
                            method='predict_proba', n_jobs=-1)[:, 1]

# --- LightGBM ---
lgb_final = LGBMClassifier(
    n_estimators     = FINAL_PARAMS.get('n_estimators', 600),
    max_depth        = FINAL_PARAMS.get('max_depth', 3),
    num_leaves       = 2 ** FINAL_PARAMS.get('max_depth', 3) - 1,
    learning_rate    = FINAL_PARAMS.get('learning_rate', 0.05),
    subsample        = FINAL_PARAMS.get('subsample', 0.75),
    colsample_bytree = FINAL_PARAMS.get('colsample_bytree', 0.65),
    reg_alpha        = FINAL_PARAMS.get('reg_alpha', 3.5),
    reg_lambda       = FINAL_PARAMS.get('reg_lambda', 8.5),
    min_child_samples= FINAL_PARAMS.get('min_child_weight', 8) * 5,
    scale_pos_weight = best_spw,
    random_state=42, n_jobs=-1, verbose=-1,
)
oof_lgb = cross_val_predict(lgb_final, Xsel, y, cv=cv,
                            method='predict_proba', n_jobs=-1)[:, 1]

# --- CatBoost ---
# FIX #4: 'subsample' is ignored unless bootstrap_type='Bernoulli' (or 'Poisson').
# CatBoost defaults to Bayesian bootstrap, which silently ignores subsample.
cat_final = CatBoostClassifier(
    iterations       = FINAL_PARAMS.get('n_estimators', 600),
    depth            = FINAL_PARAMS.get('max_depth', 3),
    learning_rate    = FINAL_PARAMS.get('learning_rate', 0.05),
    l2_leaf_reg      = FINAL_PARAMS.get('reg_lambda', 8.5),
    subsample        = FINAL_PARAMS.get('subsample', 0.75),
    bootstrap_type   = 'Bernoulli',
    scale_pos_weight = best_spw,
    random_state=42, verbose=0, allow_writing_files=False,
)
oof_cat = cross_val_predict(cat_final, Xsel, y, cv=cv,
                            method='predict_proba', n_jobs=1)[:, 1]

auc_xgb = roc_auc_score(y, oof_xgb)
auc_lgb = roc_auc_score(y, oof_lgb)
auc_cat = roc_auc_score(y, oof_cat)

print(f"  XGBoost:  {auc_xgb:.5f}")
print(f"  LightGBM: {auc_lgb:.5f}")
print(f"  CatBoost: {auc_cat:.5f}")

print(f"\n  Correlations:")
print(f"    XGB-LGB: {np.corrcoef(oof_xgb, oof_lgb)[0,1]:.4f}")
print(f"    XGB-CAT: {np.corrcoef(oof_xgb, oof_cat)[0,1]:.4f}")
print(f"    LGB-CAT: {np.corrcoef(oof_lgb, oof_cat)[0,1]:.4f}")

# --- Blend experiments ---
print("\n  Blend experiments:")

# Equal rank blend
rank_equal = (rankdata(oof_xgb) + rankdata(oof_lgb) + rankdata(oof_cat)) / 3
auc_equal = roc_auc_score(y, rank_equal)
print(f"    Equal rank-blend:     {auc_equal:.5f}")

# AUC-weighted rank blend
total_auc = auc_xgb + auc_lgb + auc_cat
w_x = auc_xgb / total_auc
w_l = auc_lgb / total_auc
w_c = auc_cat / total_auc
rank_weighted = w_x * rankdata(oof_xgb) + w_l * rankdata(oof_lgb) + w_c * rankdata(oof_cat)
auc_weighted = roc_auc_score(y, rank_weighted)
print(f"    AUC-weighted blend:   {auc_weighted:.5f}")

# XGB-heavy blend
rank_xgb_heavy = 0.5 * rankdata(oof_xgb) + 0.25 * rankdata(oof_lgb) + 0.25 * rankdata(oof_cat)
auc_xgb_heavy = roc_auc_score(y, rank_xgb_heavy)
print(f"    XGB-heavy (50/25/25): {auc_xgb_heavy:.5f}")

# Top-2 blend
aucs_dict = {'xgb': auc_xgb, 'lgb': auc_lgb, 'cat': auc_cat}
oofs_dict = {'xgb': oof_xgb, 'lgb': oof_lgb, 'cat': oof_cat}
sorted_models = sorted(aucs_dict, key=aucs_dict.get, reverse=True)
top2 = sorted_models[:2]
rank_top2 = (rankdata(oofs_dict[top2[0]]) + rankdata(oofs_dict[top2[1]])) / 2
auc_top2 = roc_auc_score(y, rank_top2)
print(f"    Top-2 ({top2[0]}+{top2[1]}):       {auc_top2:.5f}")

# Probability average
prob_blend = (oof_xgb + oof_lgb + oof_cat) / 3
auc_prob = roc_auc_score(y, prob_blend)
print(f"    Prob average:         {auc_prob:.5f}")

# Collect all results
all_results = {
    'xgb_only':       (auc_xgb,       oof_xgb),
    'lgb_only':       (auc_lgb,       oof_lgb),
    'cat_only':       (auc_cat,       oof_cat),
    'rank_equal':     (auc_equal,     rank_equal),
    'rank_weighted':  (auc_weighted,  rank_weighted),
    'rank_xgb_heavy': (auc_xgb_heavy, rank_xgb_heavy),
    'rank_top2':      (auc_top2,      rank_top2),
    'prob_blend':     (auc_prob,      prob_blend),
}

best_name = max(all_results, key=lambda k: all_results[k][0])
best_final_auc = all_results[best_name][0]
print(f"\n  Best so far: {best_name}  AUC={best_final_auc:.5f}")


# ============================================================
# 12. STEP 5: STACKING
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Stacking meta-learner")
print("=" * 60)

stack_train = pd.DataFrame({
    'xgb': oof_xgb,
    'lgb': oof_lgb,
    'cat': oof_cat,
})

# FIX #3: initialize best_meta_C so it's always defined, even if stacking
# never becomes the winning strategy.
best_meta_C = 1.0

# Try multiple meta-learners
for C_val in [0.1, 0.5, 1.0, 5.0]:
    meta = LogisticRegression(C=C_val, random_state=42, max_iter=1000)
    oof_stack = cross_val_predict(meta, stack_train, y, cv=cv,
                                  method='predict_proba')[:, 1]
    auc_stack = roc_auc_score(y, oof_stack)
    marker = ' ✓ NEW BEST' if auc_stack > best_final_auc else ''
    print(f"  LogReg C={C_val}: AUC={auc_stack:.5f}{marker}")

    if auc_stack > best_final_auc:
        best_name = 'stacked'
        best_final_auc = auc_stack
        best_meta_C = C_val
        all_results['stacked'] = (auc_stack, oof_stack)

if best_name == 'stacked':
    print(f"\n  ✓ Stacking wins! C={best_meta_C}, AUC={best_final_auc:.5f}")
else:
    print(f"\n  ✗ Stacking didn't beat {best_name} ({best_final_auc:.5f})")


# ============================================================
# 13. STEP 6: GENERATE SUBMISSION
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Generate submission")
print("=" * 60)

Xtest = test[best_cols]

# Fit all models on full training data
print("  Fitting XGBoost on full data...")
xgb_final.fit(Xsel, y)
pred_xgb = xgb_final.predict_proba(Xtest)[:, 1]

print("  Fitting LightGBM on full data...")
lgb_final.fit(Xsel, y)
pred_lgb = lgb_final.predict_proba(Xtest)[:, 1]

print("  Fitting CatBoost on full data...")
cat_final.fit(Xsel, y)
pred_cat = cat_final.predict_proba(Xtest)[:, 1]

# Generate test predictions based on best strategy
print(f"\n  Applying strategy: {best_name}")

if best_name == 'xgb_only':
    test_pred = pred_xgb

elif best_name == 'lgb_only':
    test_pred = pred_lgb

elif best_name == 'cat_only':
    test_pred = pred_cat

elif best_name == 'rank_equal':
    test_pred = (rankdata(pred_xgb) + rankdata(pred_lgb) +
                 rankdata(pred_cat)) / (3 * len(pred_xgb))

elif best_name == 'rank_weighted':
    test_pred = (w_x * rankdata(pred_xgb) + w_l * rankdata(pred_lgb) +
                 w_c * rankdata(pred_cat)) / len(pred_xgb)

elif best_name == 'rank_xgb_heavy':
    test_pred = (0.5 * rankdata(pred_xgb) + 0.25 * rankdata(pred_lgb) +
                 0.25 * rankdata(pred_cat)) / len(pred_xgb)

elif best_name == 'rank_top2':
    preds_dict = {'xgb': pred_xgb, 'lgb': pred_lgb, 'cat': pred_cat}
    test_pred = (rankdata(preds_dict[top2[0]]) +
                 rankdata(preds_dict[top2[1]])) / (2 * len(pred_xgb))

elif best_name == 'prob_blend':
    test_pred = (pred_xgb + pred_lgb + pred_cat) / 3

elif best_name == 'stacked':
    stack_test = pd.DataFrame({
        'xgb': pred_xgb, 'lgb': pred_lgb, 'cat': pred_cat
    })
    meta_final = LogisticRegression(C=best_meta_C, random_state=42, max_iter=1000)
    meta_final.fit(stack_train, y)
    test_pred = meta_final.predict_proba(stack_test)[:, 1]

# Save submission
sub = pd.DataFrame({'uid': test['uid'], 'TARGET': test_pred})
out_dir = r'E:\senior_ds_test\senior_ds_test\final_submission'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'final_submission_v2.csv')
sub.to_csv(out_path, index=False)

print(f"\n  Saved: {out_path}")
print(f"  Rows:  {len(sub)}")
print(f"\n  Prediction stats:")
print(f"    mean = {test_pred.mean():.6f}")
print(f"    std  = {test_pred.std():.6f}")
print(f"    min  = {test_pred.min():.6f}")
print(f"    max  = {test_pred.max():.6f}")
print(f"\n  Sample predictions:")
print(sub.head(10).to_string(index=False))


# ============================================================
# 14. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"""
  PREVIOUS BEST AUC:  0.68019

  FEATURE ENGINEERING:
    Total features built: {len(feature_cols)}
    Features selected:    {len(best_cols)}

  INDIVIDUAL MODELS:
    XGBoost:              {auc_xgb:.5f}
    LightGBM:             {auc_lgb:.5f}
    CatBoost:             {auc_cat:.5f}

  ENSEMBLE STRATEGIES:
    Equal rank-blend:     {auc_equal:.5f}
    AUC-weighted blend:   {auc_weighted:.5f}
    XGB-heavy (50/25/25): {auc_xgb_heavy:.5f}
    Top-2 ({top2[0]}+{top2[1]}):       {auc_top2:.5f}
    Prob average:         {auc_prob:.5f}
    Stacked:              {all_results.get('stacked', (0,))[0]:.5f}

  ════════════════════════════════════════
  🏆 BEST STRATEGY: {best_name}
  🏆 FINAL CV AUC:  {best_final_auc:.5f}
  🏆 IMPROVEMENT:   {best_final_auc - 0.68019:+.5f}
  ════════════════════════════════════════

  Submission saved to: {out_path}
""")

# ============================================================
# 15. SAVE ARTIFACTS FOR REPRODUCIBILITY
# ============================================================
print("Saving artifacts...")

# Save feature importance
imp_df = pd.DataFrame({
    'feature': imp.index,
    'importance': imp.values
}).sort_values('importance', ascending=False)
imp_df.to_csv(os.path.join(out_dir, 'feature_importance_v2.csv'), index=False)

# Save best params
import json as json_lib
params_path = os.path.join(out_dir, 'best_params_v2.json')
with open(params_path, 'w') as f:
    json_lib.dump({
        'best_strategy': best_name,
        'best_cv_auc': best_final_auc,
        'n_features_total': len(feature_cols),
        'n_features_selected': len(best_cols),
        'selected_features': best_cols,
        'xgb_params': FINAL_PARAMS,
        'individual_aucs': {
            'xgb': auc_xgb, 'lgb': auc_lgb, 'cat': auc_cat
        },
    }, f, indent=2, default=str)

# Save engineered features for future use
train[['uid', 'TARGET'] + best_cols].to_csv(
    os.path.join(out_dir, 'features_train_v2.csv'), index=False)
test[['uid'] + best_cols].to_csv(
    os.path.join(out_dir, 'features_test_v2.csv'), index=False)

print(f"  feature_importance_v2.csv")
print(f"  best_params_v2.json")
print(f"  features_train_v2.csv")
print(f"  features_test_v2.csv")
print("\nDone! ✓")

e:\senior_ds_test\senior_ds_test\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING DATA
  flag_train: (261383, 3)
  acc_train:  (1245310, 7)  (223918 uids)
  enq_train:  (1909926, 4)  (261383 uids)

Building features...

  Train: (261383, 117)
  Test:  (46127, 115)
  Features: 114
  NaN train: False
  NaN test:  False
  has_accounts mean: 0.8567

STEP 1: Baseline with all features
  All 114 features: AUC = 0.68167

  Top 25 features:
     1. acc_days_since_open_mean                 0.0693
     2. recent_enq_vs_acc                        0.0430
     3. acc_n_open                               0.0383
     4. acc_age_min                              0.0356
     5. enq_days_since_first                     0.0320
     6. microloan_x_open                         0.0320
     7. tenure_vs_enq_span                       0.0308
     8. acc_n_recent                             0.0292
     9. acc_open_ratio                           0.0256
    10. is_cash_loan                             0.0254
    11. acc_overdue_sum                          0.0239
    12. acc_overdue_m

Best trial: 6. Best value: 0.68217:  35%|███▌      | 21/60 [47:01<1:27:20, 134.38s/it]


[W 2026-09-14 20:31:24,758] Trial 21 failed with parameters: {'n_estimators': 888, 'max_depth': 5, 'learning_rate': 0.03197192621361806, 'subsample': 0.7054379146454224, 'colsample_bytree': 0.7022501082210159, 'min_child_weight': 11, 'gamma': 5.472390643029829, 'reg_alpha': 5.670396699356147, 'reg_lambda': 7.236019116158951, 'scale_pos_weight': 1.5370154916385184} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "e:\senior_ds_test\senior_ds_test\venv\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2076\2229913586.py", line 528, in objective
    oof = cross_val_predict(m, Xsel, y, cv=cv,
  File "e:\senior_ds_test\senior_ds_test\venv\lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "e:\senior_ds_test\senior_ds_test\venv\lib\site-packages\sklearn\model_selection\_validat

KeyboardInterrupt: 